In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("/kaggle/input/nasa-battery-dataset/cleaned_dataset/metadata.csv")
df1 = pd.read_csv("/kaggle/input/nasa-battery-dataset/cleaned_dataset/data/00001.csv")

In [3]:
df

,type,start_time,ambient_temperature,battery_id,test_id,uid,filename,Capacity,Re,Rct
0,discharge,[2010. 7. 21. 15. 0. ...,4,B0047,0,1,00001.csv,1.6743047446975208,NaN,NaN
1,impedance,[2010. 7. 21. 16. 53. ...,24,B0047,1,2,00002.csv,NaN,0.05605783343888099,0.20097016584458333
2,charge,[2010. 7. 21. 17. 25. ...,4,B0047,2,3,00003.csv,NaN,NaN,NaN
3,impedance,[2010 7 21 20 31 5],24,B0047,3,4,00004.csv,NaN,0.05319185850921101,0.16473399914864734
4,discharge,[2.0100e+03 7.0000e+00 2.1000e+01 2.1000e+01 2...,4,B0047,4,5,00005.csv,1.5243662105099023,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
7560,impedance,[2010. 9. 30. 7. 36. ...,24,B0055,247,7561,07561.csv,NaN,0.0968087979207628,0.15489738203707232
7561,discharge,[2010. 9. 30. 8. 8. ...,4,B0055,248,7562,07562.csv,1.0201379996149256,NaN,NaN
7562,charge,[2010. 9. 30. 8. 48. 54.25],4,B0055,249,7563,07563.csv,NaN,NaN,NaN
7563,discharge,[2010. 9. 30. 11. 50. ...,4,B0055,250,7564,07564.csv,0.9907591663373165,NaN,NaN


In [4]:
df1

,Voltage_measured,Current_measured,Temperature_measured,Current_load,Voltage_load,Time
0,4.246711,0.000252,6.212696,0.0002,0.000,0.000
1,4.246764,-0.001411,6.234019,0.0002,4.262,9.360
2,4.039277,-0.995093,6.250255,1.0000,3.465,23.281
3,4.019506,-0.996731,6.302176,1.0000,3.451,36.406
4,4.004763,-0.992845,6.361645,1.0000,3.438,49.625
...,...,...,...,...,...,...
485,3.303251,-0.001760,9.662331,0.0004,0.000,6382.063
486,3.310303,-0.000756,9.390489,0.0002,0.000,6395.547
487,3.317351,-0.003318,9.137008,0.0002,0.000,6409.063
488,3.323387,-0.002291,8.972806,0.0002,0.000,6422.625


In [5]:
import os
import pandas as pd
import numpy as np

# ======================================================
# 1) Your SoC function (unchanged)
# ======================================================

def compute_soc_precise(
    df: pd.DataFrame,
    time_col: str = "Time",
    current_col: str = "Current_measured",
    voltage_col: str = "Voltage_measured",
    current_noise_threshold: float = 1e-3,
    known_capacity_Ah: float = None,
):
    """
    Estimate State of Charge (SoC) from BMS data using coulomb counting.

    Columns used:
      - Time (s)
      - Current_measured (A), negative during discharge, positive during charge
      - Voltage_measured (V) used only for optional anchoring / sanity

    Returns:
        df_out: DataFrame with extra columns:
            - delta_t      : time step in seconds
            - I_clean      : current with noise removed
            - delta_Ah     : incremental ΔAh (+ for charging, - for discharging)
            - cum_Ah       : cumulative Ah change (relative to first sample)
            - SoC_raw      : SoC from coulomb counting before anchoring
            - SoC          : SoC after anchoring to [0,1]
            - SoC_percent  : SoC in %
        capacity_Ah: capacity used for normalization (known or estimated)
    """

    df = df.copy()
    df = df.sort_values(time_col).reset_index(drop=True)

    # 1) Time step Δt
    df["delta_t"] = df[time_col].diff().fillna(0.0)  # seconds

    # 2) Noise filter on current
    I = df[current_col].values.astype(float)
    I_clean = np.where(np.abs(I) < current_noise_threshold, 0.0, I)
    df["I_clean"] = I_clean

    # 3) Convert current*dt to Ah
    df["delta_Ah"] = df["I_clean"] * df["delta_t"] / 3600.0

    # 4) Cumulative Ah from the start of the file
    df["cum_Ah"] = df["delta_Ah"].cumsum()

    # 5) Estimate capacity if not given (from discharge part)
    if known_capacity_Ah is None:
        discharge_Ah = (
            (df["I_clean"].clip(upper=0).abs() * df["delta_t"] / 3600.0).sum()
        )
        capacity_Ah = discharge_Ah
    else:
        capacity_Ah = float(known_capacity_Ah)

    # 6) Raw SoC from coulomb counting.
    df["SoC_raw"] = 1.0 + df["cum_Ah"] / capacity_Ah

    # 7) Anchor SoC between 0 and 1 over this file
    soc0 = df["SoC_raw"].iloc[0]
    socN = df["SoC_raw"].iloc[-1]

    if soc0 != socN:
        df["SoC"] = (df["SoC_raw"] - socN) / (soc0 - socN)  # rescale to [0,1]
    else:
        df["SoC"] = df["SoC_raw"]

    df["SoC"] = df["SoC"].clip(0.0, 1.0)
    df["SoC_percent"] = df["SoC"] * 100.0

    return df, capacity_Ah


# ======================================================
# 2) Run SoC computation on all 7565 CSV files
# ======================================================

# Paths – adjust to your setup
METADATA_PATH = "/kaggle/input/nasa-battery-dataset/cleaned_dataset/metadata.csv"
DATA_DIR      = "/kaggle/input/nasa-battery-dataset/cleaned_dataset/data"
OUTPUT_DIR    = "/kaggle/working/data_with_soc"   # new folder for CSVs with SoC

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Load metadata so we know which filenames to process
metadata = pd.read_csv(METADATA_PATH)

# Optional: if the filename column is not string type, convert it
metadata["filename"] = metadata["filename"].astype(str)

# Process each file listed in metadata
for i, fname in enumerate(metadata["filename"], start=1):
    in_path = os.path.join(DATA_DIR, fname)
    out_path = os.path.join(OUTPUT_DIR, fname)

    if not os.path.isfile(in_path):
        print(f"[WARN] {fname}: file not found, skipping.")
        continue

    try:
        df = pd.read_csv(in_path)
    except Exception as e:
        print(f"[WARN] {fname}: error reading file ({e}), skipping.")
        continue

    # Make sure required columns exist
    required_cols = ["Time", "Current_measured", "Voltage_measured"]
    if not all(col in df.columns for col in required_cols):
        print(f"[WARN] {fname}: missing required columns {required_cols}, skipping.")
        continue

    # Compute SoC for this file
    df_soc, cap_Ah = compute_soc_precise(
        df,
        time_col="Time",
        current_col="Current_measured",
        voltage_col="Voltage_measured",
    )

    # df_soc already contains a "SoC" column (and SoC_percent, etc.)
    # Save to new CSV in OUTPUT_DIR
    df_soc.to_csv(out_path, index=False)

    # Progress log every 100 files
    if i % 100 == 0:
        print(f"Processed {i} files...")

print("Done. All CSVs with SoC are in:", OUTPUT_DIR)


[WARN] 00002.csv: missing required columns ['Time', 'Current_measured', 'Voltage_measured'], skipping.
[WARN] 00004.csv: missing required columns ['Time', 'Current_measured', 'Voltage_measured'], skipping.


/usr/local/lib/python3.11/dist-packages/pandas/core/computation/expressions.py:73: RuntimeWarning: invalid value encountered in greater_equal
  return op(a, b)
/usr/local/lib/python3.11/dist-packages/pandas/core/computation/expressions.py:73: RuntimeWarning: invalid value encountered in less_equal
  return op(a, b)
/usr/local/lib/python3.11/dist-packages/pandas/core/computation/expressions.py:73: RuntimeWarning: invalid value encountered in greater_equal
  return op(a, b)
/usr/local/lib/python3.11/dist-packages/pandas/core/computation/expressions.py:73: RuntimeWarning: invalid value encountered in less_equal
  return op(a, b)
/usr/local/lib/python3.11/dist-packages/pandas/core/computation/expressions.py:73: RuntimeWarning: invalid value encountered in greater_equal
  return op(a, b)
/usr/local/lib/python3.11/dist-packages/pandas/core/computation/expressions.py:73: RuntimeWarning: invalid value encountered in less_equal
  return op(a, b)
/usr/local/lib/python3.11/dist-packages/pandas/co

[WARN] 00014.csv: missing required columns ['Time', 'Current_measured', 'Voltage_measured'], skipping.
[WARN] 00016.csv: missing required columns ['Time', 'Current_measured', 'Voltage_measured'], skipping.
[WARN] 00018.csv: missing required columns ['Time', 'Current_measured', 'Voltage_measured'], skipping.
[WARN] 00020.csv: missing required columns ['Time', 'Current_measured', 'Voltage_measured'], skipping.


/usr/local/lib/python3.11/dist-packages/pandas/core/computation/expressions.py:73: RuntimeWarning: invalid value encountered in greater_equal
  return op(a, b)
/usr/local/lib/python3.11/dist-packages/pandas/core/computation/expressions.py:73: RuntimeWarning: invalid value encountered in less_equal
  return op(a, b)
/usr/local/lib/python3.11/dist-packages/pandas/core/computation/expressions.py:73: RuntimeWarning: invalid value encountered in greater_equal
  return op(a, b)
/usr/local/lib/python3.11/dist-packages/pandas/core/computation/expressions.py:73: RuntimeWarning: invalid value encountered in less_equal
  return op(a, b)
/usr/local/lib/python3.11/dist-packages/pandas/core/computation/expressions.py:73: RuntimeWarning: invalid value encountered in greater_equal
  return op(a, b)
/usr/local/lib/python3.11/dist-packages/pandas/core/computation/expressions.py:73: RuntimeWarning: invalid value encountered in less_equal
  return op(a, b)
/usr/local/lib/python3.11/dist-packages/pandas/co

[WARN] 00030.csv: missing required columns ['Time', 'Current_measured', 'Voltage_measured'], skipping.
[WARN] 00032.csv: missing required columns ['Time', 'Current_measured', 'Voltage_measured'], skipping.
[WARN] 00034.csv: missing required columns ['Time', 'Current_measured', 'Voltage_measured'], skipping.
[WARN] 00036.csv: missing required columns ['Time', 'Current_measured', 'Voltage_measured'], skipping.


...

/usr/local/lib/python3.11/dist-packages/pandas/core/computation/expressions.py:73: RuntimeWarning: invalid value encountered in greater_equal
  return op(a, b)
/usr/local/lib/python3.11/dist-packages/pandas/core/computation/expressions.py:73: RuntimeWarning: invalid value encountered in less_equal
  return op(a, b)
/usr/local/lib/python3.11/dist-packages/pandas/core/computation/expressions.py:73: RuntimeWarning: invalid value encountered in greater_equal
  return op(a, b)
/usr/local/lib/python3.11/dist-packages/pandas/core/computation/expressions.py:73: RuntimeWarning: invalid value encountered in less_equal
  return op(a, b)
/usr/local/lib/python3.11/dist-packages/pandas/core/computation/expressions.py:73: RuntimeWarning: invalid value encountered in greater_equal
  return op(a, b)
/usr/local/lib/python3.11/dist-packages/pandas/core/computation/expressions.py:73: RuntimeWarning: invalid value encountered in less_equal
  return op(a, b)


[WARN] 07531.csv: missing required columns ['Time', 'Current_measured', 'Voltage_measured'], skipping.
[WARN] 07533.csv: missing required columns ['Time', 'Current_measured', 'Voltage_measured'], skipping.
[WARN] 07535.csv: missing required columns ['Time', 'Current_measured', 'Voltage_measured'], skipping.
[WARN] 07537.csv: missing required columns ['Time', 'Current_measured', 'Voltage_measured'], skipping.


/usr/local/lib/python3.11/dist-packages/pandas/core/computation/expressions.py:73: RuntimeWarning: invalid value encountered in greater_equal
  return op(a, b)
/usr/local/lib/python3.11/dist-packages/pandas/core/computation/expressions.py:73: RuntimeWarning: invalid value encountered in less_equal
  return op(a, b)
/usr/local/lib/python3.11/dist-packages/pandas/core/computation/expressions.py:73: RuntimeWarning: invalid value encountered in greater_equal
  return op(a, b)
/usr/local/lib/python3.11/dist-packages/pandas/core/computation/expressions.py:73: RuntimeWarning: invalid value encountered in less_equal
  return op(a, b)
/usr/local/lib/python3.11/dist-packages/pandas/core/computation/expressions.py:73: RuntimeWarning: invalid value encountered in greater_equal
  return op(a, b)
/usr/local/lib/python3.11/dist-packages/pandas/core/computation/expressions.py:73: RuntimeWarning: invalid value encountered in less_equal
  return op(a, b)
/usr/local/lib/python3.11/dist-packages/pandas/co

[WARN] 07547.csv: missing required columns ['Time', 'Current_measured', 'Voltage_measured'], skipping.
[WARN] 07549.csv: missing required columns ['Time', 'Current_measured', 'Voltage_measured'], skipping.
[WARN] 07559.csv: missing required columns ['Time', 'Current_measured', 'Voltage_measured'], skipping.
[WARN] 07561.csv: missing required columns ['Time', 'Current_measured', 'Voltage_measured'], skipping.
Done. All CSVs with SoC are in: /kaggle/working/data_with_soc


/usr/local/lib/python3.11/dist-packages/pandas/core/computation/expressions.py:73: RuntimeWarning: invalid value encountered in greater_equal
  return op(a, b)
/usr/local/lib/python3.11/dist-packages/pandas/core/computation/expressions.py:73: RuntimeWarning: invalid value encountered in less_equal
  return op(a, b)
/usr/local/lib/python3.11/dist-packages/pandas/core/computation/expressions.py:73: RuntimeWarning: invalid value encountered in greater_equal
  return op(a, b)
/usr/local/lib/python3.11/dist-packages/pandas/core/computation/expressions.py:73: RuntimeWarning: invalid value encountered in less_equal
  return op(a, b)
/usr/local/lib/python3.11/dist-packages/pandas/core/computation/expressions.py:73: RuntimeWarning: invalid value encountered in greater_equal
  return op(a, b)
/usr/local/lib/python3.11/dist-packages/pandas/core/computation/expressions.py:73: RuntimeWarning: invalid value encountered in less_equal
  return op(a, b)


In [6]:
import os
import numpy as np
import pandas as pd


def compute_soc_for_cycle(
    df: pd.DataFrame,
    op_type: str,
    time_col: str = "Time",
    current_col: str = "Current_measured",
    current_noise_threshold: float = 1e-3,
    known_capacity_Ah: float = None,
):
    """
    Compute State of Charge (SoC) for a single cycle, depending on operation type.

    op_type:
      - "discharge": SoC goes from ~1 -> 0 using discharge current
      - "charge"   : SoC goes from ~0 -> 1 using charge current
      - "impedance": no time-series current; returns SoC = NaN for all rows

    For charge / discharge:
      - Uses coulomb counting: I * dt / 3600 [Ah]
      - If known_capacity_Ah is None, capacity is estimated from the cycle itself
        (sum of Ah over the active sign).

    Returns:
      df_out: original df + added columns:
          - delta_t      : time step (s)
          - I_clean      : denoised current
          - SoC          : [0, 1]
          - SoC_percent  : [0, 100]
      capacity_Ah: capacity used for normalization
    """
    op_type = str(op_type).lower()

    # -----------------------------
    # Impedance: no waveform -> NaN SoC
    # -----------------------------
    if op_type == "impedance":
        df = df.copy()
        df["SoC"] = np.nan
        df["SoC_percent"] = np.nan
        return df, np.nan

    # -----------------------------
    # Charge / Discharge
    # -----------------------------
    df = df.copy()
    df = df.sort_values(time_col).reset_index(drop=True)

    # 1) Time step
    df["delta_t"] = df[time_col].diff().fillna(0.0)  # seconds

    # 2) Clean current (remove tiny noise)
    I = df[current_col].astype(float).values
    I_clean = np.where(np.abs(I) < current_noise_threshold, 0.0, I)
    df["I_clean"] = I_clean

    # Prepare arrays for integration
    dt = df["delta_t"].values

    if op_type == "discharge":
        # Negative current = discharge
        I_dis = np.clip(I_clean, None, 0.0)  # <= 0
        delta_Ah_dis = -I_dis * dt / 3600.0   # positive Ah
        df["delta_Ah_dis"] = delta_Ah_dis

        # Estimate capacity if not given
        if known_capacity_Ah is None:
            capacity_Ah = delta_Ah_dis.sum()
        else:
            capacity_Ah = float(known_capacity_Ah)

        if capacity_Ah <= 0:
            # Fallback to avoid division by zero
            df["SoC"] = np.nan
            df["SoC_percent"] = np.nan
            return df, np.nan

        df["cum_Ah_dis"] = delta_Ah_dis.cumsum()
        # SoC ~ 1 at start, 0 at end of full discharge
        df["SoC"] = 1.0 - df["cum_Ah_dis"] / capacity_Ah

    elif op_type == "charge":
        # Positive current = charge
        I_chg = np.clip(I_clean, 0.0, None)  # >= 0
        delta_Ah_chg = I_chg * dt / 3600.0   # positive Ah
        df["delta_Ah_chg"] = delta_Ah_chg

        # Estimate capacity if not given
        if known_capacity_Ah is None:
            capacity_Ah = delta_Ah_chg.sum()
        else:
            capacity_Ah = float(known_capacity_Ah)

        if capacity_Ah <= 0:
            # Fallback to avoid division by zero
            df["SoC"] = np.nan
            df["SoC_percent"] = np.nan
            return df, np.nan

        df["cum_Ah_chg"] = delta_Ah_chg.cumsum()
        # SoC ~ 0 at start, 1 at end of full charge
        df["SoC"] = df["cum_Ah_chg"] / capacity_Ah

    else:
        # Unknown type -> no SoC
        df["SoC"] = np.nan
        df["SoC_percent"] = np.nan
        return df, np.nan

    # Clip numerical noise
    df["SoC"] = df["SoC"].clip(0.0, 1.0)
    df["SoC_percent"] = df["SoC"] * 100.0

    return df, capacity_Ah


In [7]:
import os
import pandas as pd

# ---------- Paths: adjust for your environment ----------
METADATA_PATH = "/kaggle/input/nasa-battery-dataset/cleaned_dataset/metadata.csv"
DATA_DIR      = "/kaggle/input/nasa-battery-dataset/cleaned_dataset/data"
OUTPUT_DIR    = "/kaggle/working/data_with_soc_all_types"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Load metadata so we know operation type + filename
meta = pd.read_csv(METADATA_PATH)

# Ensure columns exist
if "filename" not in meta.columns or "type" not in meta.columns:
    raise ValueError("metadata.csv must contain 'filename' and 'type' columns.")

meta["filename"] = meta["filename"].astype(str)
meta["type"] = meta["type"].astype(str)

for i, row in meta.iterrows():
    fname = row["filename"]
    op_type = row["type"]  # "charge", "discharge", or "impedance"

    in_path = os.path.join(DATA_DIR, fname)
    out_path = os.path.join(OUTPUT_DIR, fname)

    if not os.path.isfile(in_path):
        print(f"[WARN] {fname}: file not found, skipping.")
        continue

    try:
        df = pd.read_csv(in_path)
    except Exception as e:
        print(f"[WARN] {fname}: error reading file ({e}), skipping.")
        continue

    # For charge/discharge we need Time + Current_measured
    if op_type.lower() in ["charge", "discharge"]:
        required_cols = ["Time", "Current_measured"]
        if not all(c in df.columns for c in required_cols):
            print(f"[WARN] {fname}: missing {required_cols}, cannot compute SoC, skipping.")
            continue

        df_soc, cap_Ah = compute_soc_for_cycle(
            df,
            op_type=op_type,
            time_col="Time",
            current_col="Current_measured",
            # known_capacity_Ah=2.0  # <- uncomment to force rated capacity
        )

    else:
        # Impedance: no waveform; function will create NaN SoC
        df_soc, cap_Ah = compute_soc_for_cycle(df, op_type=op_type)

    # Save with SoC column
    df_soc.to_csv(out_path, index=False)

    if (i + 1) % 100 == 0:
        print(f"Processed {i + 1} / {len(meta)} files...")

print("Done. Files with SoC are in:", OUTPUT_DIR)


Processed 100 / 7565 files...
Processed 200 / 7565 files...
Processed 300 / 7565 files...
Processed 400 / 7565 files...
Processed 500 / 7565 files...
Processed 600 / 7565 files...
Processed 700 / 7565 files...
Processed 800 / 7565 files...


/tmp/ipykernel_48/4126542930.py:57: RuntimeWarning: invalid value encountered in less
  I_clean = np.where(np.abs(I) < current_noise_threshold, 0.0, I)
/usr/local/lib/python3.11/dist-packages/pandas/core/computation/expressions.py:73: RuntimeWarning: invalid value encountered in greater_equal
  return op(a, b)
/usr/local/lib/python3.11/dist-packages/pandas/core/computation/expressions.py:73: RuntimeWarning: invalid value encountered in less_equal
  return op(a, b)


Processed 900 / 7565 files...
Processed 1000 / 7565 files...


/tmp/ipykernel_48/4126542930.py:57: RuntimeWarning: invalid value encountered in less
  I_clean = np.where(np.abs(I) < current_noise_threshold, 0.0, I)
/usr/local/lib/python3.11/dist-packages/pandas/core/computation/expressions.py:73: RuntimeWarning: invalid value encountered in greater_equal
  return op(a, b)
/usr/local/lib/python3.11/dist-packages/pandas/core/computation/expressions.py:73: RuntimeWarning: invalid value encountered in less_equal
  return op(a, b)


Processed 1100 / 7565 files...
Processed 1200 / 7565 files...


...

Processed 4400 / 7565 files...


/tmp/ipykernel_48/4126542930.py:57: RuntimeWarning: invalid value encountered in less
  I_clean = np.where(np.abs(I) < current_noise_threshold, 0.0, I)
/usr/local/lib/python3.11/dist-packages/pandas/core/computation/expressions.py:73: RuntimeWarning: invalid value encountered in greater_equal
  return op(a, b)
/usr/local/lib/python3.11/dist-packages/pandas/core/computation/expressions.py:73: RuntimeWarning: invalid value encountered in less_equal
  return op(a, b)


Processed 4500 / 7565 files...
Processed 4600 / 7565 files...
Processed 4700 / 7565 files...
Processed 4800 / 7565 files...
Processed 4900 / 7565 files...
Processed 5000 / 7565 files...
Processed 5100 / 7565 files...
Processed 5200 / 7565 files...
Processed 5300 / 7565 files...
Processed 5400 / 7565 files...
Processed 5500 / 7565 files...
Processed 5600 / 7565 files...
Processed 5700 / 7565 files...
Processed 5800 / 7565 files...
Processed 5900 / 7565 files...
Processed 6000 / 7565 files...
Processed 6100 / 7565 files...
Processed 6200 / 7565 files...
Processed 6300 / 7565 files...
Processed 6400 / 7565 files...


/tmp/ipykernel_48/4126542930.py:57: RuntimeWarning: invalid value encountered in less
  I_clean = np.where(np.abs(I) < current_noise_threshold, 0.0, I)
/usr/local/lib/python3.11/dist-packages/pandas/core/computation/expressions.py:73: RuntimeWarning: invalid value encountered in greater_equal
  return op(a, b)
/usr/local/lib/python3.11/dist-packages/pandas/core/computation/expressions.py:73: RuntimeWarning: invalid value encountered in less_equal
  return op(a, b)


Processed 6500 / 7565 files...
Processed 6600 / 7565 files...
Processed 6700 / 7565 files...
Processed 6800 / 7565 files...
Processed 6900 / 7565 files...
Processed 7000 / 7565 files...
Processed 7100 / 7565 files...
Processed 7200 / 7565 files...
Processed 7300 / 7565 files...
Processed 7400 / 7565 files...
Processed 7500 / 7565 files...
Done. Files with SoC are in: /kaggle/working/data_with_soc_all_types


In [9]:
df2 = pd.read_csv("/kaggle/working/data_with_soc_all_types/00001.csv")
df2

,Voltage_measured,Current_measured,Temperature_measured,Current_load,Voltage_load,Time,delta_t,I_clean,delta_Ah_dis,cum_Ah_dis,SoC,SoC_percent
0,4.246711,0.000252,6.212696,0.0002,0.000,0.000,0.000,0.000000,-0.000000,-0.000000,1.000000,100.000000
1,4.246764,-0.001411,6.234019,0.0002,4.262,9.360,9.360,-0.001411,0.000004,0.000004,0.999998,99.999785
2,4.039277,-0.995093,6.250255,1.0000,3.465,23.281,13.921,-0.995093,0.003848,0.003852,0.997742,99.774227
3,4.019506,-0.996731,6.302176,1.0000,3.451,36.406,13.125,-0.996731,0.003634,0.007486,0.995612,99.561216
4,4.004763,-0.992845,6.361645,1.0000,3.438,49.625,13.219,-0.992845,0.003646,0.011131,0.993475,99.347516
...,...,...,...,...,...,...,...,...,...,...,...,...
485,3.303251,-0.001760,9.662331,0.0004,0.000,6382.063,13.641,-0.001760,0.000007,1.705952,0.000015,0.001528
486,3.310303,-0.000756,9.390489,0.0002,0.000,6395.547,13.484,0.000000,-0.000000,1.705952,0.000015,0.001528
487,3.317351,-0.003318,9.137008,0.0002,0.000,6409.063,13.516,-0.003318,0.000012,1.705964,0.000008,0.000798
488,3.323387,-0.002291,8.972806,0.0002,0.000,6422.625,13.562,-0.002291,0.000009,1.705973,0.000003,0.000292


In [10]:
import pandas as pd
import numpy as np
import os

# ======================================================
# Paths (adjust to your environment)
# ======================================================
METADATA_PATH = "/kaggle/input/nasa-battery-dataset/cleaned_dataset/metadata.csv"
SAVE_PATH     = "/kaggle/working/metadata_with_cycles.csv"

# ======================================================
# Load metadata
# Expected columns: type, start_time, ambient_temperature,
#                   battery_id, test_id, uid, filename, Capacity, Re, Rct, ...
# ======================================================
meta = pd.read_csv(METADATA_PATH)

# Basic sanity checks
required_cols = ["battery_id", "type", "start_time", "filename"]
missing = [c for c in required_cols if c not in meta.columns]
if missing:
    raise ValueError(f"metadata.csv is missing columns: {missing}")

# ------------------------------------------------------
# 1) Parse start_time as datetime (if possible)
#    If it’s already numeric (e.g., MATLAB datenum converted), this will still try.
# ------------------------------------------------------
meta["start_time_parsed"] = pd.to_datetime(meta["start_time"], errors="coerce")

# If some times are NaT, we still want deterministic ordering.
# We’ll sort using parsed time first, then fallback to original index.
meta["_orig_idx"] = np.arange(len(meta))

meta = meta.sort_values(
    by=["battery_id", "start_time_parsed", "_orig_idx"]
).reset_index(drop=True)

# ------------------------------------------------------
# 2) Global cycle number per battery
#    => For each battery_id, 1..N in chronological order
# ------------------------------------------------------
meta["cycle_number"] = meta.groupby("battery_id").cumcount() + 1

# ------------------------------------------------------
# 3) Cycle number per battery AND operation type
#    => For each battery_id & type (charge/discharge/impedance), 1..N
# ------------------------------------------------------
meta["cycle_number_by_type"] = meta.groupby(["battery_id", "type"]).cumcount() + 1

# (Optional) If you want separate columns for each type, you can pivot later.
# For now we keep one generic column.

# ------------------------------------------------------
# 4) Save updated metadata
# ------------------------------------------------------
# Drop helper columns if you don't want them
meta = meta.drop(columns=["start_time_parsed", "_orig_idx"])

meta.to_csv(SAVE_PATH, index=False)
print(f"Saved metadata with cycle_number columns to: {SAVE_PATH}")

# Quick peek
print(
    meta[
        [
            "battery_id",
            "type",
            "start_time",
            "filename",
            "cycle_number",
            "cycle_number_by_type",
        ]
    ]
    .head(20)
)


/tmp/ipykernel_48/2077354883.py:28: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  meta["start_time_parsed"] = pd.to_datetime(meta["start_time"], errors="coerce")


Saved metadata with cycle_number columns to: /kaggle/working/metadata_with_cycles.csv
   battery_id       type                                         start_time  \
0       B0005     charge  [2.0080e+03 4.0000e+00 2.0000e+00 1.3000e+01 8...   
1       B0005  discharge  [2.0080e+03 4.0000e+00 2.0000e+00 1.5000e+01 2...   
2       B0005     charge  [2.0080e+03 4.0000e+00 2.0000e+00 1.6000e+01 3...   
3       B0005  discharge  [2.0080e+03 4.0000e+00 2.0000e+00 1.9000e+01 4...   
4       B0005     charge  [2.0080e+03 4.0000e+00 2.0000e+00 2.0000e+01 5...   
5       B0005  discharge  [2.008e+03 4.000e+00 3.000e+00 0.000e+00 1.000...   
6       B0005     charge  [2.0080e+03 4.0000e+00 3.0000e+00 1.0000e+00 1...   
7       B0005  discharge  [2008.       4.       3.       4.      16.    ...   
8       B0005     charge  [2008.       4.       3.       5.      27.    ...   
9       B0005  discharge  [2008.       4.       3.       8.      33.    ...   
10      B0005     charge  [2008.       4.    

In [11]:
import numpy as np
import pandas as pd
from sklearn.isotonic import IsotonicRegression


def compute_soh_advanced(
    metadata: pd.DataFrame,
    rated_capacity_ah: float = 2.0,
    capacity_col: str = "Capacity",
    re_col: str = "Re",
    rct_col: str = "Rct",
) -> pd.DataFrame:
    """
    Compute an advanced SoH estimate for each row in metadata by combining:
      - Capacity-based SoH (primary, from discharge cycles)
      - BMS discharge capacity (backup when Capacity missing)
      - Impedance features (Re, Rct) normalized to early-life values
      - Monotonic smoothing per battery vs cycle_number (SoH must not increase)

    Requirements in metadata:
      - 'battery_id'
      - 'cycle_number' (global cycle index per battery)
      - 'type' (e.g. 'charge', 'discharge', 'impedance')
      - Optional:
          - capacity_col (e.g. 'Capacity')
          - 'bms_disc_capacity_ah'
          - re_col (e.g. 'Re')
          - rct_col (e.g. 'Rct')

    Returns:
      metadata_copy: same as input but with new columns:
        - 'soh_cap_raw'
        - 'soh_re_raw'
        - 'soh_rct_raw'
        - 'soh_eis_raw'
        - 'SoH'  (final smoothed SoH in [0, ~1])
    """

    df = metadata.copy()

    # ---------------------------------------------------------------------
    # 0) Sanity checks
    # ---------------------------------------------------------------------
    for col in ["battery_id", "cycle_number", "type"]:
        if col not in df.columns:
            raise ValueError(f"metadata must contain '{col}' column")

    df["type"] = df["type"].astype(str).str.lower()

    # ---------------------------------------------------------------------
    # 1) Capacity-based SoH (primary signal)
    # ---------------------------------------------------------------------
    cap_series = pd.to_numeric(df.get(capacity_col, np.nan), errors="coerce")
    df["soh_cap_raw"] = cap_series / rated_capacity_ah

    # Use BMS discharge capacity as backup for discharge cycles
    if "bms_disc_capacity_ah" in df.columns:
        bms_cap = pd.to_numeric(df["bms_disc_capacity_ah"], errors="coerce")
        mask_no_cap = df["soh_cap_raw"].isna() & df["type"].eq("discharge")
        df.loc[mask_no_cap, "soh_cap_raw"] = bms_cap[mask_no_cap] / rated_capacity_ah

    # ---------------------------------------------------------------------
    # 2) Resistance-based SoH (from Re / Rct), normalized to early-life
    # ---------------------------------------------------------------------
    # Initialize columns
    df["soh_re_raw"] = np.nan
    df["soh_rct_raw"] = np.nan

    has_re = re_col in df.columns
    has_rct = rct_col in df.columns

    if has_re:
        df[re_col] = pd.to_numeric(df[re_col], errors="coerce")
    if has_rct:
        df[rct_col] = pd.to_numeric(df[rct_col], errors="coerce")

    # Compute per-battery baselines and normalized SoH-like signals
    for bid, sub_idx in df.groupby("battery_id").groups.items():
        sub = df.loc[sub_idx]

        # Early-life baseline from first few impedance cycles
        if has_re:
            re_vals = sub[re_col].dropna()
            if not re_vals.empty:
                # baseline ~ median of first 3 valid values
                baseline_Re = re_vals.iloc[:3].median()
                # soh_re_raw = baseline_Re / Re  (higher Re => lower SoH)
                re_norm = baseline_Re / sub[re_col]
                df.loc[sub_idx, "soh_re_raw"] = re_norm.replace([np.inf, -np.inf], np.nan)

        if has_rct:
            rct_vals = sub[rct_col].dropna()
            if not rct_vals.empty:
                baseline_Rct = rct_vals.iloc[:3].median()
                rct_norm = baseline_Rct / sub[rct_col]
                df.loc[sub_idx, "soh_rct_raw"] = rct_norm.replace([np.inf, -np.inf], np.nan)

    # Combine impedance-based SoH when both or one are available
    df["soh_eis_raw"] = np.nan
    if has_re or has_rct:
        if has_re and has_rct:
            df["soh_eis_raw"] = np.nanmean(
                np.vstack([df["soh_re_raw"], df["soh_rct_raw"]]), axis=0
            )
        elif has_re:
            df["soh_eis_raw"] = df["soh_re_raw"]
        elif has_rct:
            df["soh_eis_raw"] = df["soh_rct_raw"]

    # Clip raw impedance SoH to a reasonable range
    df["soh_eis_raw"] = df["soh_eis_raw"].clip(lower=0.0, upper=1.5)

    # ---------------------------------------------------------------------
    # 3) Combine capacity + impedance into a single noisy SoH signal
    # ---------------------------------------------------------------------
    # Priority: capacity > BMS > impedance
    # We already used BMS to fill soh_cap_raw, so now:
    #   - if soh_cap_raw exists, use it (strongest)
    #   - else if soh_eis_raw exists, use that
    #   - else NaN (no signal)
    df["soh_signal"] = df["soh_cap_raw"]
    mask_no_cap_signal = df["soh_signal"].isna() & df["soh_eis_raw"].notna()
    df.loc[mask_no_cap_signal, "soh_signal"] = df.loc[mask_no_cap_signal, "soh_eis_raw"]

    # ---------------------------------------------------------------------
    # 4) Monotonic smoothing per battery vs cycle_number
    #    SoH should *not increase* with cycle_number.
    #    Use isotonic regression (increasing=False) per battery.
    # ---------------------------------------------------------------------
    df["SoH"] = np.nan

    for bid, sub_idx in df.groupby("battery_id").groups.items():
        sub = df.loc[sub_idx].sort_values("cycle_number")

        x = sub["cycle_number"].values.astype(float)
        y = sub["soh_signal"].values.astype(float)
        mask_valid = ~np.isnan(y)

        if mask_valid.sum() == 0:
            # No SoH info at all for this battery
            continue
        elif mask_valid.sum() == 1:
            # Only one labeled point: propagate that SoH to all cycles
            soh_single = y[mask_valid][0]
            df.loc[sub.index, "SoH"] = soh_single
        else:
            # Fit monotone *decreasing* curve
            ir = IsotonicRegression(increasing=False, out_of_bounds="clip")
            ir.fit(x[mask_valid], y[mask_valid])
            soh_smooth = ir.predict(x)
            df.loc[sub.index, "SoH"] = soh_smooth

    # Final clipping: SoH in [0, 1.2] (allow slight >1 due to noise)
    df["SoH"] = df["SoH"].clip(lower=0.0, upper=1.2)

    return df


In [12]:
# 1) Load your metadata that already has cycle_number etc.
meta = pd.read_csv("/kaggle/working/metadata_with_cycles.csv")

# 2) Compute advanced SoH
meta_soh = compute_soh_advanced(meta, rated_capacity_ah=2.0)

# 3) Save
meta_soh.to_csv("/kaggle/working/metadata_with_soh_advanced.csv", index=False)

print(meta_soh[["battery_id", "type", "cycle_number", "Capacity", "Re", "Rct", "SoH"]].head(30))


   battery_id       type  cycle_number            Capacity  Re  Rct       SoH
0       B0005     charge             1                 NaN NaN  NaN  0.946806
1       B0005  discharge             2  1.8564874208181574 NaN  NaN  0.946806
2       B0005     charge             3                 NaN NaN  NaN  0.946806
3       B0005  discharge             4   1.846327249719927 NaN  NaN  0.946806
4       B0005     charge             5                 NaN NaN  NaN  0.946806
5       B0005  discharge             6  1.8353491942234077 NaN  NaN  0.946806
6       B0005     charge             7                 NaN NaN  NaN  0.946806
7       B0005  discharge             8  1.8352625275821128 NaN  NaN  0.946806
8       B0005     charge             9                 NaN NaN  NaN  0.946806
9       B0005  discharge            10  1.8346455082120419 NaN  NaN  0.946806
10      B0005     charge            11                 NaN NaN  NaN  0.946806
11      B0005  discharge            12  1.8356616600675495 NaN  

/tmp/ipykernel_48/4035467779.py:102: RuntimeWarning: Mean of empty slice
  df["soh_eis_raw"] = np.nanmean(
/usr/local/lib/python3.11/dist-packages/pandas/core/computation/expressions.py:73: RuntimeWarning: invalid value encountered in greater_equal
  return op(a, b)
/usr/local/lib/python3.11/dist-packages/pandas/core/computation/expressions.py:73: RuntimeWarning: invalid value encountered in less_equal
  return op(a, b)
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).